In [ ]:
import json

from dotenv import load_dotenv
from evaluation_utils import calc_total_price
from ingest import load_faq_data
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

In [ ]:
documents = load_faq_data()

In [3]:
documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

112

In [4]:
class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate five questions this student might ask based on FAQ record. The record should contain the answer to the questions,
and the questions should be complete and not too short.
If possible, use as few words as possible from the record.

The output should resemble how people ask questions on the internet.
""".strip()

In [6]:
load_dotenv()
openai_client = OpenAI()

In [7]:
user_prompt = json.dumps(doc)

In [8]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt},
]

In [9]:
response = openai_client.responses.parse(
    model='gpt-5.4-mini',
    input=messages,
    text_format = Questions
)

In [10]:
response.output_parsed.questions

['How does the capstone homework get scored, and what parts make up the total points?',
 'What do I need to do to get all 14 points on a homework in this course?',
 'How many points are awarded for correct answers, public learning items, and adding a question to the FAQ?',
 'Is the homework score based only on quiz answers, or are there other tasks that count too?',
 'What is the maximum score for one homework assignment, and how is it calculated?']

In [11]:
!wget "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py"

--2026-07-09 18:39:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py.1’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-07-09 18:39:39 (36.8 MB/s) - ‘evaluation_utils.py.1’ saved [3073/3073]



In [18]:
from evaluation_utils import llm_structured, calc_price

In [15]:
result, usage = llm_structured(
    openai_client, data_gen_instructions, user_prompt, 
    output_type=Questions)
result.questions

['How is the capstone homework graded, and what do I need to do to get the full score?',
 'What are the point values for correct answers, public learning items, and FAQ questions in the homework scoring?',
 'How many points can I earn at most from one homework assignment in this course?',
 'Does the homework score include both answers, learning items, and an FAQ contribution? If so, how many points is each worth?',
 'What do I need to submit in a homework so that it counts toward the leaderboard, and how many points is each part worth?']

In [19]:
costs = calc_price(usage)
print(costs)

{'input_cost': 0.00018524999999999998, 'output_cost': 0.000567, 'total_cost': 0.00075225}


In [20]:
records = []
for q in result.questions:
    records.append({"question": q, "document": doc["id"]})

In [23]:
import pandas as pd

In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    
    out, usage = llm_structured_retry(
        openai_client, data_gen_instructions, user_prompt, 
        output_type=Questions)
    
    results = []
    for q in out.questions:
        results.append({"question": q, "document": doc["id"]})
    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'How are capstone homework submissions graded, and what can I do to maximize my score?',
   'document': '0d200c8c58'},
  {'question': 'What is the point breakdown for a homework assignment in the capstone project?',
   'document': '0d200c8c58'},
  {'question': 'How many points do I get for correctly answering questions, adding learning items, and contributing a FAQ question?',
   'document': '0d200c8c58'},
  {'question': 'What does the full scoring rubric look like for the homework in this course?',
   'document': '0d200c8c58'},
  {'question': 'How can I earn the maximum 14 points on a homework assignment?',
   'document': '0d200c8c58'}],
 ResponseUsage(input_tokens=247, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=346))

In [28]:
ground_truth = []
usages = []

In [29]:
for doc in tqdm(documents[:10]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/10 [00:00<?, ?it/s]

In [30]:
from concurrent.futures import ThreadPoolExecutor

In [31]:
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/1244 [00:00<?, ?it/s]

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)
    
len(ground_truth)

6220

In [34]:
total_costs = 0.0
for usage in usages:
    cost = calc_price(usage)
    total_costs = total_costs + cost["total_cost"]
    
print(total_costs)

1.025187000000001


In [36]:
calc_total_price(usages)

1.025187000000001

In [37]:
df_ground_truth = pd.DataFrame(ground_truth)

In [39]:
from pathlib import Path

df_ground_truth.to_csv("data/ground-truth-data.csv", index=False)